# VideoDB Indexing V2 Quickstart

Run the complete flow: create reusable Understanding outputs, index them, and retrieve the relevant video moments.

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## 1. Install dependencies

This installs the published VideoDB SDK and notebook dependencies.


In [ ]:
!pip install -q videodb python-dotenv


## 2. Connect to VideoDB

Set `VIDEO_DB_API_KEY` in Colab secrets/environment, or enter it when prompted.


In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()

if not os.getenv("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect(api_key=os.environ["VIDEO_DB_API_KEY"])
print("Connected to VideoDB")


## 3. Choose a video

By default, this notebook uploads the sample video used in the E2E flow: **Silicon Valley - Gilfoyle is free for hire**. To use an existing video instead, comment the upload line and uncomment the `get_video` lines in the next cell.


In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=vVlEVRKv4is"  # Silicon Valley - Gilfoyle is free for hire

collection = conn.get_collection()
video = collection.upload(VIDEO_URL)

# To use an existing video instead, comment the upload line above and uncomment these lines:
# VIDEO_ID = "m-..."
# video = collection.get_video(VIDEO_ID)

print("Collection:", collection.id)
print("Video:", video.id)
video.play()


<a id="understand"></a>
## 4. Create an Understanding run

Understanding produces reusable analyzer outputs. Here we ask for a transcript and a VLM scene description.

> [Explore the Understanding guides](understanding/README.md)


In [ ]:
understanding = video.understand(
    analyzers=[
        {
            "type": "spoken_words",
            "name": "transcript",
            "config": {"language": "en"},
        },
        {
            "type": "vlm",
            "name": "scene",
            "inputs": ["transcript"],
            "sampling": {"strategy": "uniform", "frame_count": 3},
            "config": {
                "model": "ultra",
                "prompt": (
                    "Analyze this Silicon Valley clip temporally from the sampled frame sequence. "
                    "Use the transcript when helpful. "
                    "Describe what happens in the scene and return it in the outputs field."
                ),
                "schema": {"outputs": "text"},
            },
        },
    ],
    segmentation={"type": "shot", "threshold": 30},
)

print("Understanding created")
print(f"ID: {understanding.id}")
print(f"Status: {understanding.status}")


## 5. Wait and fetch outputs

Each analyzer returns timestamped scenes with analyzer-specific data.


In [ ]:
understanding.wait_until_complete(timeout=3600, poll_interval=15)
print(f"Final status: {understanding.status}")

print("\nAnalyzer statuses:")
for analyzer in understanding.list_analyzers():
    print(f"- {analyzer.name} ({analyzer.type}): {analyzer.status}")


In [ ]:
from textwrap import fill


def as_segments(output):
    return output.get("scenes", output) if isinstance(output, dict) else output


def preview_segments(name, output, max_segments=3):
    segments = as_segments(output) or []
    label = "segment" if len(segments) == 1 else "segments"
    print(f"\n{name.title()} ({len(segments)} {label})")
    print("-" * 60)

    for segment in segments[:max_segments]:
        data = segment.get("data") or {}
        content = data.get("text") or data.get("outputs") or "(empty)"
        print(f"\n{segment.get('start')}s → {segment.get('end')}s")
        print(fill(str(content), width=100))


In [ ]:
transcript = understanding.get_analyzer("transcript").get_output()
scene = understanding.get_analyzer("scene").get_output()

preview_segments("transcript", transcript)
preview_segments("scene", scene)


<a id="index"></a>
## 6. Create indexes

An **index** makes Understanding output searchable. Configure it with:

- `use_for`: enabled retrieval methods — `semantic`, `query`, or `aggregate`.
- `fields`: optional field mappings for semantic search, filters, aggregation, and sorting.

Omit either setting to use VideoDB defaults. See the [complete indexing guide](indexing/indexing_guide.ipynb).

In [ ]:
from datetime import datetime, timezone

run_suffix = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

transcript_index_name = f"transcript_{run_suffix}"
scene_index_name = f"scene_{run_suffix}"

transcript_index = video.index(
    name=transcript_index_name,
    source=transcript
)

scene_index = video.index(
    name=scene_index_name,
    source=scene,
    use_for=["semantic", "query"],
    fields={
        "semantic": ["outputs"],
        "filter": ["outputs"],
    },
)

print("Created indexes:")
for index in (transcript_index, scene_index):
    capabilities = ", ".join(index.use_for or []) or "none"
    print(f"- {index.name}")
    print(f"  ID: {index.index_id}")
    print(f"  Status: {index.status}")
    print(f"  Capabilities: {capabilities}")


## 7. Wait for indexes

Search works best after indexes are ready.


In [ ]:
# Index builds are asynchronous — wait with the SDK's built-in helper.
# Semantic indexes flip from `building` to `ready`.
print("Index build statuses:")
for index in (transcript_index, scene_index):
    index.wait_until_complete(timeout=900, poll_interval=10)
    print(f"- {index.name}: {index.status}")
    if not index.is_successful:
        print(f"  Build failed: {index.error}")


## 8. Inspect indexed fields

Each index declares which fields can be used for semantic search, filtering, and aggregation.

In [ ]:
print("Index field mappings:")
for index in (transcript_index, scene_index):
    print(f"\n{index.name}")
    for group, fields in sorted(index.fields.items()):
        field_names = ", ".join(fields) if fields else "none"
        print(f"- {group}: {field_names}")


<a id="retrieve"></a>
## Choosing a retrieval method

Each retrieval method answers a different question:

| Method | Answers | How |
|---|---|---|
| `video.search(query=...)` | "find moments about X" in natural language — VideoDB plans across indexes | semantic + filters |
| `video.semantic_search(query=..., index_names=[...])` | "find talk **about** X" on chosen indexes | vector similarity |
| `video.query(index_name=..., filter=...)` | exact structured lookup | field filters |
| `video.aggregate(index_name=..., group_by=...)` | counts and facets | group-by |
| `video.ask(question=...)` | a synthesized answer with cited sources | retrieval + LLM |

See the [complete Search V2 guide](search/search_guide.ipynb).

## 9. Search naturally

`video.search(...)` returns a `SearchResponse`. For shot results, `response.results` is a `SearchResult` with `compile()` and `play()` methods; `play()` compiles automatically.


In [ ]:
search_response = video.search(
    query="a gift arrives at the front door",
    top_k=5,
    mode="default",
    return_fields="all",
)

if not search_response:
    raise RuntimeError("No search results found")

print(f"Found {len(search_response)} results")
print("Preparing playback...")
search_response.results.play()


## 10. Search a specific semantic index

Use `video.semantic_search(...)` when you want semantic retrieval over a known index.

In [ ]:
semantic_results = video.semantic_search(
    query="a man drops an electronic device in a trash bin",
    index_names=[scene_index_name],
    top_k=5,
    score_threshold=0.2,
    return_fields="all",
)

if not semantic_results:
    raise RuntimeError("No semantic search results found")

print(f"Found {len(semantic_results)} results")
print("Preparing playback...")
semantic_results.play()


## 11. Ask questions with sources

Use `video.ask(...)` when you want an answer grounded in retrieved video moments.

In [ ]:
answer = video.ask(
    question="what does the character say after he changes his linkedin status",
    top_k=15,
    mode="default",
    include_sources=True,
)

print("Answer")
print("-" * 60)
print(answer.answer)

if not answer.sources:
    raise RuntimeError("No playable sources were returned")

print(f"\nSources: {len(answer.sources)}")
print("Preparing the first source...")
answer.sources[0].play()


## 12. Query exact scene text with AND / OR filters

Use `video.query(...)` when you know the exact indexed field/filter.

In [ ]:
delivery_query_results = video.query(
    index_name=scene_index_name,
    filter=[{"field": "outputs", "op": "contains", "value": "front door"}],
    limit=10,
    return_fields="all",
)

if not delivery_query_results:
    raise RuntimeError("No delivery query results found")

print(f"Found {len(delivery_query_results)} results")
print("Preparing playback...")
delivery_query_results.play()


In [ ]:
trash_where = {
    "and": [
        {
            "or": [
                {"field": "outputs", "op": "contains", "value": "dustbin"},
                {"field": "outputs", "op": "contains", "value": "trash"},
            ]
        },
        {
            "or": [
                {"field": "outputs", "op": "contains", "value": "white cord"},
                {"field": "outputs", "op": "contains", "value": "wired device"},
                {"field": "outputs", "op": "contains", "value": "electronic device"},
                {"field": "outputs", "op": "contains", "value": "USB missile launcher"},
            ]
        },
    ]
}

trash_query_results = video.query(
    index_name=scene_index_name,
    filter=trash_where,
    limit=10,
    return_fields="all",
)

if not trash_query_results:
    raise RuntimeError("No trash query results found")

print(f"Found {len(trash_query_results)} results")
print("Preparing playback...")
trash_query_results.play()


## 13. Optional cleanup

Only run deletion when you are done with these preview records.


In [ ]:
DELETE_INDEXES = False

if DELETE_INDEXES:
    video.delete_index(index_id=transcript_index.index_id)
    video.delete_index(index_id=scene_index.index_id)
    print("Deleted indexes")
else:
    print("Skipping delete. Set DELETE_INDEXES=True to delete these indexes.")


In [ ]:
DELETE_UNDERSTANDING = False

if DELETE_UNDERSTANDING:
    understanding.delete()
    print("Deleted", understanding.id)
else:
    print("Skipping delete. Set DELETE_UNDERSTANDING=True to delete this Understanding.")